# Segmentation Model 3

## Основные улучшения и стратегия

### 1. Архитектура и выбор Энкодера
В этой итерации мы отошли от стандартных решений в сторону более глубокого анализа текстурных признаков:
* **Модель:** `DeepLabV3Plus`. Благодаря модулю **ASPP** (Atrous Spatial Pyramid Pooling), она эффективно работает с объектами разных масштабов, что критично, когда продукт занимает всего 10% кадра.
* **Энкодер:** `tu-convnext_base`. Механизм **Split-Attention** внутри блоков ResNeSt позволяет модели лучше разделять похожие визуальные сигналы (например, серый металл фона и корпус продукта).
* **Регуляризация:** В сегментационную голову добавлен слой `Dropout2d(0.05)` для предотвращения переобучения на малом датасете (~2000 изображений).

### 2. Продвинутая функция потерь (Combined Loss)
Вместо стандартного BCE используется взвешенная комбинация трех лоссов для стабильной сходимости:
$$Loss = 0.4 \cdot Dice + 0.4 \cdot Lovasz + 0.2 \cdot Focal$$
* **Lovasz Loss:** Напрямую оптимизирует метрику IoU (Jaccard index), сглаживая дискретную задачу сегментации.
* **Focal Loss:** Заставляет модель фокусироваться на «тяжелых» пикселях (границах и бликах на металле), не давая фону доминировать в градиентах.

### 3. Геометрические аугментации и TTA
Упор сделан на сохранение инвариантности к поворотам и заслонениям:
* **Train:** Интенсивный набор из `HorizontalFlip`, `VerticalFlip`, `RandomRotate90` и `ShiftScaleRotate` (вращение до 15°). Для борьбы с шумом используется `CoarseDropout` (имитация частичных перекрытий объекта).
* **Inference (TTA):** Применяется ансамбль из **6 трансформаций** (Original, HFlip, VFlip, Rot90, Rot270, DiagonalFlip). Усреднение предсказаний по этим осям позволяет «сгладить» маску и повысить итоговый Dice Score.

### 4. Оптимизация обучения
* **SWA (Stochastic Weight Averaging):** Начиная с 21-й эпохи (70% пути), включается усреднение весов модели. Это позволяет найти более широкие локальные минимумы и улучшить обобщающую способность на тесте.
* **Scheduler:** `CosineAnnealingLR` обеспечивает плавное снижение скорости обучения до перехода в режим SWA.
* **Hardware:** Использование **AMP (Automatic Mixed Precision)** и оптимизаций `cudnn.benchmark` + `TF32` позволило тренировать тяжелый энкодер на разрешении `352x352` без потери производительности.

### 5. Пайплайн и Submission
* **Интеграция с Drive:** Реализован автоматический бэкап лучших чекпоинтов и истории обучения (CSV) с уникальными временными метками.
* **Формат:** Маски сериализуются в JSON-списки с восстановлением оригинального разрешения через бинарный порог (`THRESHOLD = 0.5`).
* **Submission:** Автоматическая отправка результата в соревнование через Kaggle API.

---

### Итоговый конфиг:
* **Optimizer:** `AdamW` (LR=5e-4, WD=5e-4)
* **Batch Size:** 16
* **Epochs:** 30
* **Encoder:** `tu-convnext_base` (ImageNet pretrained)

# Segmentation model 3

## Init

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
import os
from google.colab import userdata

# Получаем учетные данные из менеджера секретов Colab
kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key = userdata.get('KAGGLE_KEY')

# Устанавливаем переменные окружения для Kaggle
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key
os.environ['KAGGLEHUB_CACHE'] = '/'

# Создаем директорию .kaggle, если ее нет
!mkdir -p ~/.kaggle

# Создаем файл kaggle.json с учетными данными
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')

# Устанавливаем правильные права доступа для файла kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# Теперь вы можете использовать kaggle CLI или kagglehub.login()
# kagglehub.login() не всегда нужен, если kaggle.json настроен правильно
print("Kaggle API credentials configured successfully!")

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

dl_lab_3_product_segmentation_path = kagglehub.competition_download('dl-lab-3-product-segmentation')
# packagemanager_pm_113281741_at_03_23_2026_18_04_38_path = kagglehub.notebook_output_download('packagemanager/pm-113281741-at-03-23-2026-18-04-38')

print('Data source import complete.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Путь к папке на Google Drive
DRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/segmentation_checkpoints")
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Checkpoints will be saved to: {DRIVE_CHECKPOINT_DIR}")

In [ ]:
pip install segmentation-models-pytorch timm opencv-python pandas

In [ ]:
# pip uninstall -y torch torchvision

In [ ]:
# pip install --no-cache-dir torch==2.7.0 torchvision==0.22.0 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
import os
import random
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn

## Preparation / train

### GPU speed optimization

In [ ]:
import torch

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')

print(" CUDA optimizations enabled:")
print(" cudnn.benchmark = True")
print(" TF32 matmul enabled")
print(" high precision matmul")

### CONFIG

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
from datetime import datetime

DATA_ROOT = Path(
    r"/competitions/dl-lab-3-product-segmentation/train"
)
IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR = DATA_ROOT / "masks"
SAVE_DIR = Path("./seg_train_runs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_PATH = Path("/content/drive/MyDrive/KaggleLab/dataset.csv")

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/segmentation_checkpoints") / TIMESTAMP
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BACKUP_TO_DRIVE = True

IMG_SIZE = 352  # Оптимальный размер для DeepLabV3Plus
BATCH_SIZE = 16
NUM_EPOCHS = 30
LR = 5e-4
WEIGHT_DECAY = 5e-4

NUM_WORKERS = 2
PREFETCH_FACTOR = 2
PERSISTENT_WORKERS = True

SEED = 42
THRESHOLD = 0.5

MODEL_NAME = "DeepLabV3Plus"

ENCODER_NAME = "tu-convnext_base"  # Изменено на tu-convnext_base
ENCODER_WEIGHTS = "imagenet"

SWA_START = int(NUM_EPOCHS * 0.7)  # 70% эпох
SWA_LR = LR * 0.1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

from segmentation_models_pytorch.encoders import encoders
print(f"Available encoders: {list(encoders.keys())[:20]}...")
print(f"Using MODEL: {MODEL_NAME}")
print(f"Using ENCODER: {ENCODER_NAME}")

In [ ]:
from segmentation_models_pytorch.encoders import encoders
print(encoders.keys())

In [ ]:
import shutil

def backup_checkpoint_to_drive(local_path: Path, drive_path: Path, fold: int, epoch: int, is_best: bool = False):
    if not BACKUP_TO_DRIVE or not is_best:
        return

    try:
        drive_fold_dir = drive_path / f"fold_{fold}"
        drive_fold_dir.mkdir(parents=True, exist_ok=True)

        drive_file = drive_fold_dir / "best.pth"
        shutil.copy2(local_path, drive_file)

        meta_file = drive_fold_dir / f"checkpoint_metadata.txt"
        with open(meta_file, 'a') as f:
            f.write(f"Epoch {epoch}: BEST - {local_path.name}\n")

        print(f"  Backed up BEST model to Drive: {drive_file}")
    except Exception as e:
        print(f"  Failed to backup to Drive: {e}")

### Utils

In [ ]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_image_for_stem(images_dir: Path, stem: str) -> Path | None:
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"]:
        p = images_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None


def dice_score_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > THRESHOLD).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)
    denom = preds.sum(dim=1) + targets.sum(dim=1)

    dice = (2.0 * intersection + eps) / (denom + eps)
    return dice.mean().item()


def iou_score_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > THRESHOLD).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1) - intersection

    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()


seed_everything(SEED)


### Loss

In [ ]:
dice_loss_fn = smp.losses.DiceLoss(mode='binary', from_logits=True)
focal_loss_fn = smp.losses.FocalLoss(mode='binary')
lovasz_loss_fn = smp.losses.LovaszLoss(mode='binary', from_logits=True)

def combined_loss(logits, targets):
    """
    Loss = 0.4 * Dice + 0.4 * Lovasz + 0.2 * Focal
    """
    dice = dice_loss_fn(logits, targets)
    focal = focal_loss_fn(logits, targets)
    lovasz = lovasz_loss_fn(logits, targets)

    return 0.4 * dice + 0.4 * lovasz + 0.2 * focal

### Augmentations

In [ ]:
import albumentations as A

def get_train_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.CoarseDropout(
            max_holes=8,
            max_height=img_size//30,
            max_width=img_size//30,
            fill_value=0,
            mask_fill_value=0,
            p=0.1
        ),
    ], p=1.0)

def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
    ], p=1.0)

### Dataset

In [ ]:
class BinarySegDataset(Dataset):
    def __init__(
        self,
        images_dir: Path,
        masks_dir: Path,
        img_size: int = 352,
        encoder_name: str = "resnet34",
        encoder_weights: str | None = "imagenet",
        transforms=None,
        cache_images: bool = True,
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.img_size = img_size
        self.transforms = transforms
        self.cache_images = cache_images
        self.image_cache = {}
        self.mask_cache = {}

        self.preprocess_input = None
        if encoder_weights is not None:
            self.preprocess_input = get_preprocessing_fn(
                encoder_name, pretrained=encoder_weights
            )

        self.samples = []
        for mask_path in sorted(self.masks_dir.glob("*.png")):
            stem = mask_path.stem
            image_path = find_image_for_stem(self.images_dir, stem)
            if image_path is not None:
                self.samples.append((image_path, mask_path))

        if not self.samples:
            raise RuntimeError(f"No paired samples found in {self.images_dir}")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        image_path, mask_path = self.samples[idx]

        if self.cache_images and idx in self.image_cache:
            image_rgb = self.image_cache[idx].copy()
            mask = self.mask_cache[idx].copy()
        else:
            image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
            image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

            if self.cache_images:
                self.image_cache[idx] = image_rgb.copy()
                self.mask_cache[idx] = mask.copy()

        if self.transforms is not None:
            augmented = self.transforms(image=image_rgb, mask=mask)
            image_rgb = augmented["image"]
            mask = augmented["mask"]
        else:
            image_rgb = cv2.resize(image_rgb, (self.img_size, self.img_size))
            mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        image_rgb = image_rgb.astype(np.float32)
        if self.preprocess_input is not None:
            image_rgb = self.preprocess_input(image_rgb)
        else:
            image_rgb = image_rgb / 255.0

        mask = (mask > 0).astype(np.float32)
        image = torch.from_numpy(image_rgb.transpose(2, 0, 1)).float()
        mask = torch.from_numpy(mask[None, ...]).float()

        return image, mask

### Model

In [ ]:
def build_model() -> nn.Module:
    if MODEL_NAME == "DeepLabV3Plus":
        model = smp.DeepLabV3Plus(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None
        )
        # Добавлен Dropout2d в segmentation head для регуляризации
        model.segmentation_head = nn.Sequential(
            model.segmentation_head,
            nn.Dropout2d(0.05)
        )
        print("DeepLabV3Plus with Dropout2d(0.05) in segmentation head")

    elif MODEL_NAME == "UnetPlusPlus":
        model = smp.UnetPlusPlus(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
            decoder_attention_type="scse"
        )

        for m in model.decoder.modules():
            if isinstance(m, nn.Conv2d):
                m.add_module("dropout", nn.Dropout2d(0.1))

    else:
        model = getattr(smp, MODEL_NAME)(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
        )

    return model

print(f"Model: {MODEL_NAME} with encoder: {ENCODER_NAME}")

### TRAIN / VAL LOOPS

In [ ]:
from tqdm.auto import tqdm

scaler = torch.amp.GradScaler('cuda')

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()

    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    pbar = tqdm(loader, desc="Training", leave=False)

    for images, masks in pbar:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss = loss_fn(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        running_dice += dice_score_from_logits(logits.detach(), masks)
        running_iou += iou_score_from_logits(logits.detach(), masks)

        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'dice': f'{running_dice/(pbar.n+1):.4f}'
        })

    n = len(loader)
    return {
        "loss": running_loss / n,
        "dice": running_dice / n,
        "iou": running_iou / n,
    }

@torch.no_grad()
def validate_one_epoch(model, loader, loss_fn, device):
    model.eval()

    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    pbar = tqdm(loader, desc="Validation", leave=False)

    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device)

        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss = loss_fn(logits, masks)

        running_loss += loss.item()
        running_dice += dice_score_from_logits(logits, masks)
        running_iou += iou_score_from_logits(logits, masks)

    n = len(loader)
    return {
        "loss": running_loss / n,
        "dice": running_dice / n,
        "iou": running_iou / n,
    }

### Split

In [ ]:
import pandas as pd
import torch
import os

split_df = pd.read_csv(SPLIT_PATH)

train_base_ds = BinarySegDataset(
    IMAGES_DIR, MASKS_DIR, IMG_SIZE,
    ENCODER_NAME, ENCODER_WEIGHTS,
    transforms=get_train_transforms(IMG_SIZE)
)

val_base_ds = BinarySegDataset(
    IMAGES_DIR, MASKS_DIR, IMG_SIZE,
    ENCODER_NAME, ENCODER_WEIGHTS,
    transforms=get_val_transforms(IMG_SIZE)
)

sample_stems = [p[0].stem for p in train_base_ds.samples]
stem_to_idx = {stem: i for i, stem in enumerate(sample_stems)}

img_col = 'image' if 'image' in split_df.columns else split_df.columns[0]
sub_col = 'subset' if 'subset' in split_df.columns else 'split'

train_idx = []
val_idx = []

for _, row in split_df.iterrows():
    raw_val = str(row[img_col])
    stem = os.path.splitext(os.path.basename(raw_val))[0]

    if stem in stem_to_idx:
        idx = stem_to_idx[stem]
        if str(row[sub_col]).lower() == 'train':
            train_idx.append(idx)
        elif str(row[sub_col]).lower() in ['val', 'validation', 'test']:
            val_idx.append(idx)

train_dataset = torch.utils.data.Subset(train_base_ds, train_idx)
val_dataset = torch.utils.data.Subset(val_base_ds, val_idx)

print(f"Successfully matched samples from CSV: {SPLIT_PATH}")
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

### Loader

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=PREFETCH_FACTOR,
    persistent_workers=PERSISTENT_WORKERS,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=PREFETCH_FACTOR,
    persistent_workers=PERSISTENT_WORKERS
)

### Model + Optimizer

In [ ]:
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn

model = build_model().to(DEVICE)

swa_model = AveragedModel(model)
swa_start = SWA_START

loss_fn = combined_loss

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS
)

swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR)

print(f"Optimizer: AdamW (LR={LR}, WD={WEIGHT_DECAY})")
print(f"Scheduler: CosineAnnealingLR (T_max={NUM_EPOCHS})")
print(f"SWA: start at epoch {swa_start}, SWA_LR={SWA_LR}")

### Training loop

In [ ]:
best_val_dice = -1.0
best_swa_dice = -1.0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    if epoch > swa_start:
        update_bn(train_loader, swa_model, device=DEVICE)

    train_metrics = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
    val_metrics = validate_one_epoch(model, val_loader, loss_fn, DEVICE)

    if epoch > swa_start:
        swa_model.update_parameters(model)
        swa_scheduler.step()
    else:
        scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]

    row = {
        "epoch": epoch,
        "lr": current_lr,
        "train_loss": train_metrics["loss"],
        "train_dice": train_metrics["dice"],
        "train_iou": train_metrics["iou"],
        "val_loss": val_metrics["loss"],
        "val_dice": val_metrics["dice"],
        "val_iou": val_metrics["iou"],
    }
    history.append(row)

    print(
        f"Epoch {epoch:03d}/{NUM_EPOCHS} | "
        f"Train: loss={row['train_loss']:.4f}, dice={row['train_dice']:.4f} | "
        f"Val: loss={row['val_loss']:.4f}, dice={row['val_dice']:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    if row["val_dice"] > best_val_dice:
        best_val_dice = row["val_dice"]
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": {"MODEL_NAME": MODEL_NAME, "ENCODER_NAME": ENCODER_NAME, "IMG_SIZE": IMG_SIZE},
        }, SAVE_DIR / "best.pth")
        print(f"  --> New best model saved! (Dice: {best_val_dice:.4f})")

    if epoch > swa_start:
        swa_val_metrics = validate_one_epoch(swa_model, val_loader, loss_fn, DEVICE)
        print(f"  SWA Val: dice={swa_val_metrics['dice']:.4f}, iou={swa_val_metrics['iou']:.4f}")

        if swa_val_metrics["dice"] > best_swa_dice:
            best_swa_dice = swa_val_metrics["dice"]
            torch.save({
                "model_state_dict": swa_model.state_dict(),
                "config": {"MODEL_NAME": MODEL_NAME, "ENCODER_NAME": ENCODER_NAME, "IMG_SIZE": IMG_SIZE},
            }, SAVE_DIR / "best_swa.pth")
            print(f"  --> New BEST SWA saved! (Dice: {best_swa_dice:.4f})")

torch.save({
    "model_state_dict": swa_model.state_dict(),
    "config": {
        "MODEL_NAME": MODEL_NAME,
        "ENCODER_NAME": ENCODER_NAME,
        "IMG_SIZE": IMG_SIZE
    },
}, SAVE_DIR / "swa_model.pth")

print(f"\nTraining completed!")
print(f"Best validation Dice: {best_val_dice:.4f}")
print(f"Best SWA Dice: {best_swa_dice:.4f}")

history_df = pd.DataFrame(history)
history_df.to_csv("history.csv", index=False)

if BACKUP_TO_DRIVE:
    shutil.copy2("history.csv", DRIVE_CHECKPOINT_DIR / "history.csv")
    print(f"History backed up to Drive: {DRIVE_CHECKPOINT_DIR}")

## Submission formation

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn


In [ ]:
def load_checkpoint_from_drive(fold: int, checkpoint_type: str = "swa_model"):

    drive_fold_dir = DRIVE_CHECKPOINT_DIR / f"fold_{fold}"
    checkpoint_path = drive_fold_dir / f"{checkpoint_type}.pth"

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    print(f"Loading from Drive: {checkpoint_path}")
    return torch.load(checkpoint_path, map_location="cpu")

### Config

In [ ]:
TEST_IMAGES_DIR = Path(
    r"/competitions/dl-lab-3-product-segmentation/test_images"
)
OUTPUT_CSV = "submission.csv"

# CHECKPOINT_PATH = Path(r"./seg_train_runs/best.pth")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
# THRESHOLD = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


### HELPERS

In [ ]:
import cv2
import numpy as np
import json
from pathlib import Path
import segmentation_models_pytorch as smp

def cv2_imread_unicode(path: Path, flags=cv2.IMREAD_COLOR):
    data = np.fromfile(str(path), dtype=np.uint8)
    if data.size == 0:
        return None
    return cv2.imdecode(data, flags)

def build_model_inference(model_name: str, encoder_name: str, encoder_weights=None):
    if model_name == "Unet":
        model = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif model_name == "UnetPlusPlus":
        model = smp.UnetPlusPlus(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif model_name == "DeepLabV3":
        model = smp.DeepLabV3(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif model_name == "DeepLabV3Plus":
        model = smp.DeepLabV3Plus(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
        )
        model.segmentation_head = nn.Sequential(
            model.segmentation_head,
            nn.Dropout2d(0.1)
        )
    elif model_name == "FPN":
        model = smp.FPN(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
        )
    else:
        raise ValueError(f"Unsupported MODEL_NAME: {model_name}")
    return model

def serialize_mask(mask2d: np.ndarray) -> str:
    return json.dumps(mask2d.astype(np.uint8).tolist(), separators=(",", ":"))

### Load checkpoint



In [ ]:
def load_model_from_ckpt(path):
    ckpt = torch.load(path, map_location=DEVICE)

    model = build_model_inference(
        model_name=ckpt["config"]["MODEL_NAME"],
        encoder_name=ckpt["config"]["ENCODER_NAME"],
        encoder_weights=None
    )

    state_dict = ckpt["model_state_dict"]
    new_state_dict = {}

    for k, v in state_dict.items():
        name = k.replace("module.", "") if k.startswith("module.") else k
        if name != "n_averaged":
            new_state_dict[name] = v

    model.load_state_dict(new_state_dict)
    model.to(DEVICE)
    model.eval()

    return model

swa_path = SAVE_DIR / "best_swa.pth"
best_path = SAVE_DIR / "best.pth"

if swa_path.exists():
    print("Loading SWA model...")
    model = load_model_from_ckpt(swa_path)
    model_type = "SWA"
else:
    print("Loading best model...")
    model = load_model_from_ckpt(best_path)
    model_type = "Best"

print(f"Loaded single {model_type} model for inference with TTA")

### Threshold tuning

In [ ]:
from scipy.optimize import minimize_scalar

@torch.no_grad()
def find_optimal_threshold(model, val_loader, device, bounds=(0.25, 0.7)):
    model.eval()

    all_probs = []
    all_masks = []

    print(f"Collecting TTA predictions on {len(val_loader)} batches...")

    for images, masks in tqdm(val_loader, desc="TTA collection"):
        images = images.to(device)

        # TTA: 6 transformations
        # 1. Original
        p1 = torch.sigmoid(model(images))
        # 2. Horizontal Flip
        p2 = torch.flip(torch.sigmoid(model(torch.flip(images, [3]))), [3])
        # 3. Vertical Flip
        p3 = torch.flip(torch.sigmoid(model(torch.flip(images, [2]))), [2])
        # 4. Rotate 90
        p4 = torch.rot90(torch.sigmoid(model(torch.rot90(images, 1, [2, 3]))), k=-1, dims=[2, 3])
        # 5. Rotate 270
        p5 = torch.rot90(torch.sigmoid(model(torch.rot90(images, 3, [2, 3]))), k=-3, dims=[2, 3])
        # 6. Diagonal Flip
        inp_diag = images.permute(0, 1, 3, 2)
        p6 = torch.sigmoid(model(inp_diag)).permute(0, 1, 3, 2)

        # Average TTA predictions
        tta_prob = (p1 + p2 + p3 + p4 + p5 + p6) / 6.0

        all_probs.append(tta_prob.cpu().numpy())
        all_masks.append(masks.numpy())

    print("Concatenating results...")
    probs = np.concatenate(all_probs, axis=0)
    masks = np.concatenate(all_masks, axis=0)

    def calculate_dice(threshold):
        preds = (probs > threshold).astype(np.float32)
        intersection = (preds * masks).sum(axis=(1, 2, 3))
        denom = preds.sum(axis=(1, 2, 3)) + masks.sum(axis=(1, 2, 3))
        dice_scores = (2.0 * intersection + 1e-7) / (denom + 1e-7)
        return dice_scores.mean()

    def objective(threshold):
        return -calculate_dice(threshold)

    print("Optimizing threshold...")
    result = minimize_scalar(
        objective,
        bounds=bounds,
        method='bounded',
        options={'xatol': 0.005}
    )

    optimal_threshold = result.x
    best_dice = -result.fun

    print(f"\n{'Threshold':<12} {'Dice':<10}")
    print("-" * 25)

    for t in np.arange(optimal_threshold - 0.05, optimal_threshold + 0.06, 0.01):
        t = np.clip(t, 0.1, 0.9)
        dice = calculate_dice(t)
        marker = "  ← OPTIMAL" if abs(t - optimal_threshold) < 0.001 else ""
        print(f"{t:<12.3f} {dice:<10.4f}{marker}")

    return optimal_threshold, best_dice


In [ ]:
val_loader_safe = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,  # Важно для TTA
    pin_memory=True
)

print("\nSearching optimal threshold with TTA...")
optimal_thresh, val_dice_optimized = find_optimal_threshold(
    model, val_loader_safe, DEVICE, bounds=(0.3, 0.6)
)

print(f"\nOptimal threshold: {optimal_thresh:.3f}")
print(f"Optimized Dice with TTA: {val_dice_optimized:.4f}")

### INFERENCE

In [ ]:
ckpt = torch.load(SAVE_DIR / "best.pth" if not (SAVE_DIR / "best_swa.pth").exists() else SAVE_DIR / "best_swa.pth", map_location='cpu')

image_paths = sorted([
    p for p in TEST_IMAGES_DIR.rglob("*")
    if p.suffix.lower() in IMG_EXTS
])

rows = []

THRESHOLD = optimal_thresh

preprocess_input = get_preprocessing_fn(
    ckpt['config']['ENCODER_NAME'],
    pretrained='imagenet'
)

print(f"Running inference with TTA (6 transformations)")
print(f"  - Model: {model_type}")
print(f"  - Threshold: {THRESHOLD:.3f}")

with torch.no_grad():
    for i, img_path in enumerate(image_paths, 1):
        img_bgr = cv2_imread_unicode(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            continue

        H, W = img_bgr.shape[:2]
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        inp_orig = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR).astype(np.float32)
        if preprocess_input:
            inp_orig = preprocess_input(inp_orig)
        inp = torch.from_numpy(inp_orig.transpose(2,0,1)).float().unsqueeze(0).to(DEVICE)

        model.eval()

        # 1. Original
        p1 = torch.sigmoid(model(inp))[0,0].cpu().numpy()

        # 2. Horizontal Flip
        p2 = np.flip(torch.sigmoid(model(torch.flip(inp, [3])))[0,0].cpu().numpy(), axis=1)

        # 3. Vertical Flip
        p3 = np.flip(torch.sigmoid(model(torch.flip(inp, [2])))[0,0].cpu().numpy(), axis=0)

        # 4. Rotate 90
        p4 = np.rot90(torch.sigmoid(model(torch.rot90(inp, 1, [2, 3])))[0,0].cpu().numpy(), k=-1)

        # 5. Rotate 270
        p5 = np.rot90(torch.sigmoid(model(torch.rot90(inp, 3, [2, 3])))[0,0].cpu().numpy(), k=-3)

        # 6. Diagonal Flip
        inp_diag = inp.permute(0, 1, 3, 2)
        p6 = torch.sigmoid(model(inp_diag))[0,0].cpu().numpy().T

        pred = (p1 + p2 + p3 + p4 + p5 + p6) / 6.0

        if pred.shape != (H, W):
            pred = cv2.resize(pred, (W, H))

        mask = (pred > THRESHOLD).astype(np.uint8)
        rows.append({'ImageId': img_path.name, 'mask': serialize_mask(mask)})

        if i % 50 == 0:
            print(f'Processed {i}/{len(image_paths)} images')

print(f'Processed {len(image_paths)}/{len(image_paths)} images')

submission_df = pd.DataFrame(rows)
submission_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSubmission saved to {OUTPUT_CSV}")
print(f"  Total images processed: {len(rows)}")
print(f"  Model: {model_type} with TTA (6 transformations)")
print(f"  Threshold: {THRESHOLD:.3f}")

### Backup to drive

In [ ]:
import shutil

if SAVE_DIR.exists() and DRIVE_CHECKPOINT_DIR.exists():
    files_to_backup = ["best.pth", "best_swa.pth", "swa_model.pth", "history.csv"]

    for file_name in files_to_backup:
        file_path = SAVE_DIR / file_name
        if file_path.exists():
            shutil.copy2(file_path, DRIVE_CHECKPOINT_DIR / file_name)
            print(f" {file_name} backed up to {DRIVE_CHECKPOINT_DIR}")
        else:
            print(f" {file_name} not found")
else:
    print("Error: SAVE_DIR or DRIVE_CHECKPOINT_DIR do not exist")

## Submit to kaggle

In [ ]:
!kaggle competitions submit -c dl-lab-3-product-segmentation -f submission.csv -m "DeepLabV3Plus + ConvNeXt-Base + SWA + TTA(6) + Combined Loss"

import os
print('Execution completed. Disconnecting environment...')
os._exit(0)